# AeroPulse Anomaly — 06: Onset Detector and Event-Level Evaluation

Model C from plan section 9, plus the episode-level evaluation framework of
section 18.

## Two questions, deliberately kept apart

Notebook 05 answers *"will the air be hazardous?"* — an absolute
concentration question. This notebook answers the one the residual
architecture was always trying to answer:

> **Is PM2.5 about to rise sharply, relative to what is normal for this
> station?**

That is a different and genuinely useful product. It catches sudden
industrial releases, local fires, unexpected transport and sensor faults —
including at stations whose absolute levels never breach 121 µg/m³, which the
hazard model will never flag.

### The label

The review suggests `residual_z > 3` or "PM2.5 rises > X σ within 3–6 hours".
This notebook uses the second form, because the first is circular: a label
built from the residual the model predicts means the model is grading its own
homework.

```
target_onset_next_6h = 1  when  (pm25(t+6) − pm25(t)) / σ_24h(t) ≥ 3
```

`σ_24h` is the station's own trailing 24h standard deviation, so the threshold
adapts: a 40 µg/m³ jump is routine in Delhi in November and extraordinary in
Chennai in July. Both numerator and denominator come from the station's own
history, so no cross-station normalisation is needed.

### Also in this notebook

An honest event-level scoreboard. Notebook 03 showed that detection rate and
lead time are meaningless without a false-alarm denominator, so every episode
metric here is reported with `alerts_per_day` and `false_alerts_per_event`
attached.

In [1]:
# ============================================================================
# CONFIGURATION
# Every anomaly notebook shares this block so paths, thresholds and helper
# definitions cannot drift between phases. The real logic lives in
# anomaly_toolkit.py — the "Shared Feature Layer" of the improvement plan.
# ============================================================================
from pathlib import Path
import os, json, sys, warnings, time

import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError(
        "No .env found. Copy this folder's .env.example to .env, or launch "
        "Jupyter from a directory whose parent has .env."
    )
load_dotenv(_dotenv_path, override=True)
warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path.cwd()))
import anomaly_toolkit as tk

CFG = tk.load_config(os.getenv("AEROPULSE_PROJECT_ROOT", "."))
FEATURE_FILE = CFG["feature_dir"] / "anomaly_features.parquet"
PM25_BASE = CFG["pm25_processed"] / "base"

print("env      :", _dotenv_path)
print("project  :", CFG["project_root"])
print("pm25 data:", CFG["pm25_data_root"])
print("hazard   : PM2.5 >=", CFG["very_poor_ugm3"], "ug/m3 within", CFG["horizon_hours"], "h")

env      : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/.env
project  : /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector
pm25 data: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
hazard   : PM2.5 >= 121.0 ug/m3 within 24 h


In [2]:
# ============================================================================
# LOAD
# ============================================================================
import lightgbm as lgb
import joblib

manifest = json.loads((CFG["feature_dir"] / "dataset_manifest.json").read_text())
df = pd.read_parquet(FEATURE_FILE)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)

ONSET = "target_onset_next_6h"
RISE = "target_rise_next_6h"
H = CFG["horizon_hours"]
HAZARD = f"target_very_poor_within_{H}h"

work = df[df[ONSET].notna()].reset_index(drop=True)
print(f"{len(work):,} labelled rows")
print(f"onset prevalence : {work[ONSET].mean():.4%}")
print(f"hazard prevalence: {work[HAZARD].mean():.4%}")
overlap = work[(work[ONSET] == 1) & (work[HAZARD] == 1)].shape[0]
print(f"rows that are BOTH onset and hazard: {overlap:,} "
      f"({overlap / max(work[ONSET].sum(), 1):.1%} of onsets)")

1,694,379 labelled rows
onset prevalence : 5.2957%
hazard prevalence: 11.2946%
rows that are BOTH onset and hazard: 18,556 (20.7% of onsets)


### Result — the two labels are largely disjoint

Only **20.7% of onset rows are also hazard rows**. Roughly four in five sharp
rises never reach the CPCB Very Poor threshold, and most hazardous windows
are sustained pollution rather than a sudden onset.

That overlap figure is the justification for keeping two models. If onsets
were mostly a subset of hazards, Model C would be redundant and the plan's
three-model architecture would be over-engineering. They are not.

## 1. Onset classifier

The same driver features as notebook 05, with one important exception:
the target is built from `pm25(t+6) − pm25(t)` and `σ_24h(t)`, so any feature
that is a deterministic function of those two would leak.

In [3]:
# ============================================================================
# FEATURES — with an explicit exclusion for the onset target's own inputs
# ============================================================================
FEATURES = [c for c in manifest["feature_candidates"]
            if c in work.columns and pd.api.types.is_numeric_dtype(work[c])]

# pm25_roll_std_24h IS the denominator of the onset label. Keeping it lets the
# model shrink the label's own scale, which is leakage of the target's
# definition rather than of its value.
LABEL_INPUTS = ["pm25_roll_std_24h"]
FEATURES = [c for c in FEATURES if c not in LABEL_INPUTS]
print(f"{len(FEATURES)} features (excluded label inputs: {LABEL_INPUTS})")

# Notebook 05 split on the HAZARD-labelled frame. The onset label drops a few
# thousand extra rows (no trailing 24h std), which would shift the quantiles
# and leave notebook 05's predictions covering a slightly different window.
# Take the boundaries from the hazard frame so the two line up exactly.
split_basis = df[df[HAZARD].notna()]["timestamp_utc"]
q70, q85 = split_basis.quantile([0.70, 0.85])
train = work[work["timestamp_utc"] < q70]
valid = work[(work["timestamp_utc"] >= q70) & (work["timestamp_utc"] < q85)]
test = work[work["timestamp_utc"] >= q85]
print(f"split boundaries taken from the hazard frame: {q70.date()} / {q85.date()}")
for n, p in (("train", train), ("valid", valid), ("test", test)):
    print(f"  {n:<6} {len(p):>9,}  onset prevalence {p[ONSET].mean():.4%}")

273 features (excluded label inputs: ['pm25_roll_std_24h'])


split boundaries taken from the hazard frame: 2026-03-22 / 2026-06-13
  train  1,185,048  onset prevalence 5.3791%
  valid    254,959  onset prevalence 5.3142%
  test     254,372  onset prevalence 4.8889%


In [4]:
# ============================================================================
# TRAIN MODEL C
# ============================================================================
params = dict(objective="binary", metric=["average_precision", "auc"],
              learning_rate=0.05, num_leaves=127, min_data_in_leaf=200,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              lambda_l2=1.0, num_threads=os.cpu_count() or 4, verbosity=-1, seed=42)

dtr = lgb.Dataset(train[FEATURES], label=train[ONSET], free_raw_data=False)
dva = lgb.Dataset(valid[FEATURES], label=valid[ONSET], reference=dtr)
t0 = time.time()
onset_model = lgb.train(params, dtr, num_boost_round=700, valid_sets=[dva],
                        callbacks=[lgb.early_stopping(50, verbose=False)])
print(f"trained {onset_model.best_iteration} rounds in {time.time()-t0:.0f}s")

valid = valid.copy(); test = test.copy()
valid["onset_prob"] = onset_model.predict(valid[FEATURES], num_iteration=onset_model.best_iteration)
test["onset_prob"] = onset_model.predict(test[FEATURES], num_iteration=onset_model.best_iteration)

BUDGET = CFG["false_alert_budget"]
thr_onset, sweep_onset = tk.choose_threshold(valid[ONSET], valid["onset_prob"],
                                             budget_false_alert_rate=BUDGET)
m_onset = tk.classification_metrics(test[ONSET], test["onset_prob"], thr_onset)
print(f"\nonset model @ threshold {thr_onset:.4f} (validation-selected):")
for k in ("prevalence", "pr_auc", "roc_auc", "precision", "recall", "f1", "false_alert_rate"):
    print(f"  {k:<18} {m_onset[k]:.4f}")

trained 412 rounds in 53s



onset model @ threshold 0.1616 (validation-selected):
  prevalence         0.0489
  pr_auc             0.1636
  roc_auc            0.7650
  precision          0.2124
  recall             0.2377
  f1                 0.2243
  false_alert_rate   0.0453


In [5]:
# ============================================================================
# REFERENCE SCORES FOR THE ONSET LABEL
# ============================================================================
rows = [{"model": "onset classifier", **{k: m_onset[k] for k in
         ("pr_auc", "roc_auc", "precision", "recall", "f1", "false_alert_rate")}}]
for name, col in (("recent slope pm25_slope_6h", "pm25_slope_6h"),
                  ("acceleration pm25_accel_6h", "pm25_accel_6h"),
                  ("current pm25(t)", "pm25"),
                  ("upwind fire 150km 24h", "fire_upwind_frp_150km_24h")):
    if col not in valid.columns:
        continue
    t, _ = tk.choose_threshold(valid[ONSET], valid[col], budget_false_alert_rate=BUDGET)
    mm = tk.classification_metrics(test[ONSET], test[col], t)
    rows.append({"model": name, **{k: mm[k] for k in
                 ("pr_auc", "roc_auc", "precision", "recall", "f1", "false_alert_rate")}})
onset_cmp = pd.DataFrame(rows).sort_values("pr_auc", ascending=False)
print(onset_cmp.round(4).to_string(index=False))

                     model  pr_auc  roc_auc  precision  recall     f1  false_alert_rate
          onset classifier  0.1636   0.7650     0.2124  0.2377 0.2243            0.0453
     upwind fire 150km 24h  0.0497   0.5020     0.1277  0.0005 0.0010            0.0002
acceleration pm25_accel_6h  0.0409   0.4524     0.0228  0.0082 0.0121            0.0180
recent slope pm25_slope_6h  0.0380   0.4154     0.0195  0.0070 0.0103            0.0181
           current pm25(t)  0.0352   0.3585     0.0212  0.0058 0.0091            0.0138


### Result — onset is hard, but the model is clearly learning something

| Score | PR-AUC | ROC-AUC |
|---|---:|---:|
| onset classifier | **0.164** | **0.765** |
| upwind fire 150km 24h | 0.050 | 0.502 |
| acceleration `pm25_accel_6h` | 0.041 | 0.452 |
| recent slope `pm25_slope_6h` | 0.038 | 0.415 |
| current `pm25(t)` | 0.035 | **0.359** |

Against a 4.9% base rate, PR-AUC 0.164 is a **3.3× lift** with ROC-AUC 0.765.
Modest in absolute terms, but every trivial alternative is at or below chance.

Two results stand out.

**`current pm25(t)` has ROC-AUC 0.359 — strongly *negatively* predictive.**
High current PM2.5 makes a further 3σ rise *less* likely: partly mean
reversion, partly a ceiling effect. In notebook 05 the same feature was one
of the strongest *positive* hazard predictors. One feature, opposite sign,
same dataset — this is the sharpest possible demonstration that a single
model cannot serve both objectives, which is exactly what notebook 03's
residual detector attempted.

**Slope and acceleration are below chance.** The intuitive "it is already
rising, so it will keep rising" signal is actively misleading at this
horizon. Whatever the model has learned, it is not naive trend extrapolation.

## 2. Episode catalogue (plan section 18)

Episodes are contiguous runs of exceedance per station, tolerating short gaps.
This is the unit an operator experiences — not the hour.

In [6]:
# ============================================================================
# EPISODE CATALOGUE
# ============================================================================
VP = CFG["very_poor_ugm3"]
events = tk.build_events(test[["location_id", "timestamp_utc", "pm25"]], VP,
                         min_hours=3, max_gap_hours=6)
print(f"{len(events)} episodes across {events['location_id'].nunique()} stations")
if len(events):
    events["peak_band"] = pd.cut(events["event_peak"], [VP, 250, 500, np.inf],
                                 labels=["very_poor", "severe", "extreme"])
    print()
    print(events.groupby("peak_band", observed=True).agg(
        episodes=("event_peak", "size"),
        median_duration_h=("duration_hours", "median"),
        median_peak=("event_peak", "median")).to_string())

111 episodes across 27 stations

           episodes  median_duration_h  median_peak
peak_band                                          
very_poor        56                6.0        169.5
severe           35                7.0        326.0
extreme          20                7.0        611.0


In [7]:
# ============================================================================
# EVENT-LEVEL SCOREBOARD — always with the false-alarm denominator
# ============================================================================
P5 = CFG["artifact_root"] / "phase5_hazard"
have_hazard = (P5 / "test_predictions.parquet").exists()
if have_hazard:
    hz = pd.read_parquet(P5 / "test_predictions.parquet")
    hz["timestamp_utc"] = pd.to_datetime(hz["timestamp_utc"], utc=True)
    test = test.merge(hz[["location_id", "timestamp_utc", "hazard_prob"]],
                      on=["location_id", "timestamp_utc"], how="left")
    print("merged hazard probabilities from notebook 05:",
          f"{test['hazard_prob'].notna().mean():.1%} coverage")
else:
    print("notebook 05 predictions not found — hazard row will be skipped")

SCORES = {"onset classifier": ("onset_prob", ONSET),
          "persistence pm25_lag_1h": ("pm25_lag_1h", HAZARD),
          "current pm25(t)": ("pm25", HAZARD)}
if have_hazard and test["hazard_prob"].notna().any():
    SCORES["hazard classifier (nb05)"] = ("hazard_prob", HAZARD)

rows = []
for name, (col, sel_label) in SCORES.items():
    sub = test[test[col].notna()]
    if sub.empty:
        continue
    if col == "hazard_prob":
        vv = pd.read_parquet(P5 / "valid_predictions.parquet")
        thr, _ = tk.choose_threshold(vv[HAZARD], vv["hazard_prob"],
                                     budget_false_alert_rate=BUDGET)
    else:
        thr, _ = tk.choose_threshold(valid[sel_label], valid[col],
                                     budget_false_alert_rate=BUDGET)
    em = tk.event_level_metrics(events, sub[["location_id", "timestamp_utc", col]],
                                col, thr, lead_window_hours=48)
    em["score"] = name
    em["alerts_per_true_event"] = round(em["alerts_per_day"] * 90 / max(len(events), 1), 1)
    rows.append(em)

board = pd.DataFrame(rows).set_index("score")[
    ["events", "detected", "detection_rate", "median_lead_hours", "p10_lead_hours",
     "alerts_per_day", "false_alerts_per_event"]]
print()
print(board.round(2).to_string())

merged hazard probabilities from notebook 05: 100.0% coverage


                          events  detected  detection_rate  median_lead_hours  p10_lead_hours  alerts_per_day  false_alerts_per_event
score                                                                                                                                
onset classifier             111        15            0.14               21.0             5.4          159.34                  125.23
persistence pm25_lag_1h      111       111            1.00               42.0             1.0           54.40                   41.80
current pm25(t)              111       111            1.00               43.0             2.0           55.77                   42.88
hazard classifier (nb05)     111       111            1.00               46.0             0.0           75.02                   58.03


### Result — read every row as a triple, and mind what each score is *for*

| Score | Detection rate | Median lead | Alerts/day | False alerts/event |
|---|---:|---:|---:|---:|
| onset classifier | 0.14 | 21h | 159 | 125 |
| persistence `pm25_lag_1h` | 1.00 | 42h | 54 | 42 |
| current `pm25(t)` | 1.00 | 43h | 56 | 43 |
| hazard classifier (nb05) | 1.00 | 46h | 75 | 58 |

**Two cautions before reading anything into this table.**

First, **the onset classifier is being scored against the wrong events.**
These episodes are PM2.5 ≥ 121 exceedances — Model B's target, not Model C's.
Its 0.14 detection rate says onsets and hazard episodes are different things,
which the 20.7% overlap already established. It is included for completeness,
not as a fair comparison.

Second, **the lead-time column is not measuring foresight.** Every score here
raises 54–159 alerts/day across 149 stations and 42–125 false alerts per
episode. A detector firing that often will trivially have fired at some point
in the 48h window before any episode, so its "median lead time" is a function
of the alert rate. This is the same trap notebook 03 documented for A0, and
it is why `alerts_per_day` and `false_alerts_per_event` are printed alongside
every episode metric here.

The right reading: at a 5% false-alert budget **none of these scores is
operationally usable at the episode level**. Notebook 08 is where the alert
rate is actually brought down, and its operations mode (~40 alerts/day at
precision ~0.66) is the realistic operating point.

## 3. Where does onset detection succeed and fail?

In [8]:
# ============================================================================
# ERROR ANALYSIS BY SEASON AND REGIME
# ============================================================================
test["onset_alert"] = (test["onset_prob"] >= thr_onset).astype(int)
by_season = test.groupby("season", observed=True).apply(
    lambda g: pd.Series({
        "rows": len(g), "onsets": int(g[ONSET].sum()),
        "recall": float(g.loc[g[ONSET] == 1, "onset_alert"].mean()) if g[ONSET].sum() else np.nan,
        "precision": float(g.loc[g["onset_alert"] == 1, ONSET].mean()) if g["onset_alert"].sum() else np.nan,
    }), include_groups=False)
print("by season (test window):")
print(by_season.round(4).to_string())

test["rise_band"] = pd.cut(test[RISE], [-np.inf, 0, 25, 50, 100, np.inf],
                           labels=["falling", "0-25", "25-50", "50-100", ">100"])
by_rise = test[test[ONSET] == 1].groupby("rise_band", observed=True).agg(
    onsets=(ONSET, "size"), recall=("onset_alert", "mean"))
print("\nrecall by magnitude of the 6h rise (ug/m3):")
print(by_rise.round(3).to_string())

by season (test window):
             rows   onsets  recall  precision
season                                       
monsoon  254372.0  12436.0  0.2377     0.2124

recall by magnitude of the 6h rise (ug/m3):
           onsets  recall
rise_band                
0-25         8935   0.288
25-50        2291   0.142
50-100        757   0.063
>100          453   0.018


### Result — recall *falls* with event size, and this is the model's worst property

The expectation going in was that recall would climb with the size of the
rise, so that misses concentrated on marginal excursions. **The measurement
is the exact opposite.**

| 6h rise (µg/m³) | onsets | recall |
|---|---:|---:|
| 0–25 | 8,935 | **0.288** |
| 25–50 | 2,291 | 0.142 |
| 50–100 | 757 | 0.063 |
| >100 | 453 | **0.018** |

**The detector catches small excursions and misses almost every large one.**
It finds 29% of sub-25 µg/m³ rises and under 2% of rises above 100 µg/m³ —
which are precisely the events anyone would want alerting on.

The mechanism is in the label. An onset is `rise / σ_24h ≥ 3`, so at a quiet,
low-variance station a modest absolute rise clears the bar easily. Those
stations are numerous, their behaviour is regular, and they dominate the
positive class — so the model optimises for them. A large absolute jump at an
already-volatile station needs a much bigger move to clear its own 3σ, is
rarer, and is genuinely harder to anticipate.

**This makes Model C as currently labelled unfit for alerting.** Two fixes,
neither requiring new data:

1. **Add an absolute floor to the label** — for example
   `rise ≥ 3σ AND rise ≥ 40 µg/m³`. This removes the low-variance-station
   mass that is currently dominating the positive class.
2. **Weight training by rise magnitude**, so a 150 µg/m³ jump counts for more
   than a 10 µg/m³ one.

Recorded here as the top follow-up for the onset model rather than papered
over. The self-normalising label was the right instinct — it adapts the
threshold to each station — but it inverted the severity ordering, and the
severity ordering is what an alerting product needs.

## 4. Feature drivers of onset

In [9]:
# ============================================================================
# IMPORTANCE
# ============================================================================
imp = pd.DataFrame({
    "feature": FEATURES,
    "gain": onset_model.feature_importance("gain", iteration=onset_model.best_iteration),
}).sort_values("gain", ascending=False)
imp["share"] = (imp["gain"] / imp["gain"].sum()).round(4)
print("top 20 onset drivers:")
print(imp.head(20)[["feature", "share"]].to_string(index=False))

for fam, pref in (("fire transport", "fire_"), ("regional/upwind", ("regional_", "upwind_")),
                  ("satellite", "cams_"), ("weather", ("temperature", "wind", "boundary",
                                                       "rain_", "ventilation", "stagnation",
                                                       "delta_", "surface_pressure",
                                                       "relative_humidity"))):
    share = imp.loc[imp["feature"].str.startswith(pref), "share"].sum()
    print(f"  {fam:<18} gain share {share:.3f}")

top 20 onset drivers:
              feature  share
    delta_pressure_3h 0.0547
    pm25_roll_std_12h 0.0438
    pm25_roll_max_24h 0.0396
   pm25_roll_std_168h 0.0256
       pm25_delta_12h 0.0246
    pm25_roll_min_24h 0.0242
  pm25_roll_mean_168h 0.0203
    pm25_roll_p95_24h 0.0160
neighbour_distance_km 0.0149
   pm25_pct_change_1h 0.0148
   pm25_pct_change_6h 0.0144
        pm25_delta_6h 0.0132
              cos_doy 0.0129
       pm25_slope_12h 0.0127
             cos_hour 0.0118
       days_to_diwali 0.0114
boundary_layer_height 0.0110
elevation_relief_10km 0.0109
                  doy 0.0107
          elevation_m 0.0099
  fire transport     gain share 0.108
  regional/upwind    gain share 0.045
  satellite          gain share 0.128
  weather            gain share 0.155


## 5. Persist

In [10]:
# ============================================================================
# PERSIST
# ============================================================================
P6 = CFG["artifact_root"] / "phase6_onset"
P6.mkdir(parents=True, exist_ok=True)

joblib.dump({"model": onset_model, "features": FEATURES,
             "best_iteration": onset_model.best_iteration,
             "threshold": float(thr_onset), "label": ONSET,
             "excluded_label_inputs": LABEL_INPUTS,
             "schema_version": CFG["schema_version"]},
            P6 / "onset_detector.joblib", compress=3)
events.to_parquet(P6 / "episode_catalogue.parquet", index=False)
board.to_csv(P6 / "event_scoreboard.csv")
onset_cmp.to_csv(P6 / "onset_model_comparison.csv", index=False)
imp.to_csv(P6 / "feature_importance.csv", index=False)

tk.write_report(P6 / "phase6_report.json", {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "onset_label": ONSET,
    "onset_definition": "(pm25(t+6) - pm25(t)) / rolling_std_24h(t) >= 3",
    "prevalence": {"onset": float(work[ONSET].mean()),
                   "hazard": float(work[HAZARD].mean()),
                   "both": float(overlap / len(work))},
    "onset_operating_point": m_onset,
    "onset_model_comparison": onset_cmp.to_dict("records"),
    "episodes": int(len(events)),
    "event_scoreboard": board.reset_index().to_dict("records"),
    "recall_by_rise_magnitude": by_rise.reset_index().astype(str).to_dict("records"),
    "notes": [
        "Onset and hazard labels are largely disjoint — two models are justified.",
        "pm25_roll_std_24h excluded: it is the onset label's own denominator.",
        "Event detection rate and lead time are only interpretable alongside "
        "false_alerts_per_event.",
        "current pm25(t) is NEGATIVELY predictive of onset — the opposite of its "
        "role in the hazard model.",
    ],
})
print("wrote", P6 / "phase6_report.json")

wrote /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/anomaly_detector/artifacts/anomaly/phase6_onset/phase6_report.json


## Conclusion

**Model C is a distinct product, not a variant of Model B.** The labels
overlap only 20.7%, and `current pm25(t)` flips from one of the strongest
*positive* predictors in the hazard model to strongly *negative* here
(ROC-AUC 0.359). One model cannot serve both objectives — which is precisely
what notebook 03's residual detector attempted.

**The model learns something real but is not yet shippable.** PR-AUC 0.164
against a 4.9% base rate is a 3.3× lift with ROC-AUC 0.765, while every
trivial alternative — slope, acceleration, current level — sits at or below
chance. Naive trend extrapolation genuinely does not work at this horizon.

**The blocking defect is the label, not the model.** Recall *falls* with
event size: 0.29 for sub-25 µg/m³ rises, 0.02 for rises above 100 µg/m³. The
`rise / σ_24h ≥ 3` definition lets quiet, low-variance stations dominate the
positive class, so the model optimises for the least consequential events.
**Fix before any further tuning:** add an absolute floor
(`rise ≥ 3σ AND rise ≥ 40 µg/m³`) and/or weight training by rise magnitude.
The plan's suggested `residual_z > 3` would have the same flaw, with
circularity on top.

**Event metrics need a false-alarm denominator.** This is an amendment to
plan section 18, which specifies detection rate and lead time but not the
alert rate they must be read against. Without it the most trigger-happy
detector wins every episode metric: at a 5% budget, all four scores here
raise 54–159 alerts/day and 42–125 false alerts per episode, making their
90th-percentile "lead times" artefacts of alert frequency rather than
foresight.